In [ ]:
# Install the project dependencies required by this notebook.
%pip install -q pandas numpy duckdb pyarrow scikit-learn xgboost mlflow matplotlib scipy joblib pyyaml

In [ ]:
# Mount Google Drive so Colab can access the private MIMIC-IV files and derived artifacts.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Clone or update the GitHub repository so the notebook can import the shared project code.
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/mbakos95/aki-sentinel.git"
REPO_DIR = Path("/content/aki-sentinel")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

sys.path.insert(0, str(REPO_DIR))

In [ ]:
# Define the private MIMIC-IV and artifact locations used by the pipeline.
from pathlib import Path

MIMIC_ROOT = Path("/content/drive/MyDrive/MIMIC-IV")
HOSP_DIR = MIMIC_ROOT / "hosp"
ICU_DIR = MIMIC_ROOT / "icu"

PRIVATE_ROOT = Path("/content/drive/MyDrive/AKI-Sentinel-Private")
ARTIFACT_DIR = PRIVATE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Import the trained model, evaluation helpers, and custom drift-monitoring utilities.
import json
import joblib
import pandas as pd

from src.evaluation import classification_metrics, save_json
from src.modeling import get_model_columns
from src.monitoring import numeric_drift_table

In [ ]:
# Load the historical reference cohort, later current cohort, trained XGBoost model, and validation-selected threshold.
SPLIT_DIR = ARTIFACT_DIR / "splits"
MODEL_DIR = ARTIFACT_DIR / "models"
EVAL_DIR = ARTIFACT_DIR / "evaluation"

reference_df = pd.read_parquet(SPLIT_DIR / "train.parquet")
current_df = pd.read_parquet(SPLIT_DIR / "test.parquet")
model = joblib.load(MODEL_DIR / "xgboost.joblib")

thresholds = json.loads((EVAL_DIR / "thresholds.json").read_text())
xgb_threshold = thresholds["xgboost"]

In [ ]:
# Measure feature-level distribution and missingness drift between the historical reference and later cohort.
feature_columns, numeric_columns, categorical_columns = get_model_columns(reference_df)

feature_drift = numeric_drift_table(
    reference_df,
    current_df,
    numeric_columns,
)

feature_drift.head(20)

In [ ]:
# Compare prediction distributions and outcome performance between the historical reference and later current cohort.
reference_prob = model.predict_proba(reference_df[feature_columns])[:, 1]
current_prob = model.predict_proba(current_df[feature_columns])[:, 1]

reference_metrics = classification_metrics(
    reference_df["target"],
    reference_prob,
    xgb_threshold,
)
current_metrics = classification_metrics(
    current_df["target"],
    current_prob,
    xgb_threshold,
)

monitoring_metrics = {
    "reference": reference_metrics,
    "current": current_metrics,
    "prediction_mean_reference": float(reference_prob.mean()),
    "prediction_mean_current": float(current_prob.mean()),
    "drifted_numeric_features": int(feature_drift["drift_flag"].sum()),
    "numeric_features_monitored": int(len(feature_drift)),
}

monitoring_metrics

In [ ]:
# Create a compact monitoring status from performance degradation and the number of drifted features.
sensitivity_drop = (
    reference_metrics["sensitivity"]
    - current_metrics["sensitivity"]
)

if current_metrics["sensitivity"] < 0.80 or sensitivity_drop >= 0.10:
    status = "CRITICAL"
elif feature_drift["drift_flag"].sum() >= 5:
    status = "WARNING"
else:
    status = "HEALTHY"

monitoring_metrics["status"] = status
monitoring_metrics["sensitivity_drop"] = float(sensitivity_drop)

monitoring_metrics

In [ ]:
# Save aggregate monitoring outputs for the dashboard without exporting patient-level MIMIC data.
MONITOR_DIR = ARTIFACT_DIR / "monitoring"
MONITOR_DIR.mkdir(parents=True, exist_ok=True)

feature_drift.to_csv(
    MONITOR_DIR / "feature_drift.csv",
    index=False,
)
save_json(
    monitoring_metrics,
    MONITOR_DIR / "monitoring_metrics.json",
)